## Experiment Vases and Faces

In [ ]:
%matplotlib widget

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

image_path = Path("faces_vase_color.jpg")
if not image_path.exists():
    image_path = Path.cwd() / "faces_vase_color.jpg"
if not image_path.exists():
    raise FileNotFoundError("Could not find faces_vase_color.jpg next to the notebook.")

original = np.asarray(Image.open(image_path).convert("RGB"))
rng = np.random.default_rng()
clicks = []
click_count = 0

fig, ax = plt.subplots(figsize=(8, 6))
fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False
ax.set_axis_off()
image_artist = ax.imshow(original)
score_text = ax.text(
    0.02, 0.98, "Clicks: 0 | Noise: 0%",
    transform=ax.transAxes,
    va="top",
    color="white",
    fontsize=12,
    bbox={"facecolor": "black", "alpha": 0.65, "pad": 5},
)
ax.set_title("Click anywhere on the image — each click adds salt-and-pepper noise")

def add_noise(image, amount):
    noisy = image.copy()
    mask = rng.random(image.shape[:2]) < amount
    salt = rng.random(image.shape[:2]) < 0.5
    noisy[mask & salt] = 255
    noisy[mask & ~salt] = 0
    return noisy

def on_click(event):
    global click_count
    if event.inaxes is not ax or event.xdata is None or event.ydata is None:
        return

    click_count += 1
    clicks.append((event.xdata, event.ydata))
    noise_amount = min(0.45, click_count * 0.015)
    image_artist.set_data(add_noise(original, noise_amount))
    score_text.set_text(
        f"Clicks: {click_count} | Noise: {noise_amount:.1%}"
    )
    ax.plot(event.xdata, event.ydata, "r+", ms=12, mew=2)
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("button_press_event", on_click)
plt.show()
